# Custom steering vectors - BAGEL
Build text-derived directions, then edit image generation. Use an A100 80 GB.


## 1. Setup
Publish this branch before opening Colab. Run setup in a fresh runtime.


In [ ]:
from pathlib import Path

PROJECT = Path("/content/bagel-space")
UPSTREAM = Path("/content/BAGEL")
if not PROJECT.exists():
    !git clone --branch experiment/custom-steering-vectors https://github.com/Nicerova7/bagel-space.git /content/bagel-space
if not UPSTREAM.exists():
    !git clone https://github.com/ByteDance-Seed/Bagel.git /content/BAGEL
!git -C /content/bagel-space fetch origin
!git -C /content/bagel-space checkout experiment/custom-steering-vectors
!git -C /content/BAGEL checkout a2fa77dd8caeefc41e6607ae0ec17408d3f4ee9f
%pip install -r /content/bagel-space/requirements.txt


### Install FlashAttention
Install the Astral binary for Python 3.13, PyTorch 2.11, and CUDA 13.0.


In [ ]:
%pip install --no-deps "https://github.com/astral-sh-build/build-flash-attention/releases/download/v2.8.3-r4/flash_attn-2.8.3%2Bcu.13.0.torch.2.11-cp313-cp313-manylinux_2_24_x86_64.whl"
print("Restart the session, then continue below.")


## 2. Load BAGEL
Restart after installation, then run once.


In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

import torch
from IPython.display import display

PROJECT = Path("/content/bagel-space")
UPSTREAM = Path("/content/BAGEL")
os.chdir(PROJECT)
os.environ["PYTHONPATH"] = str(UPSTREAM) + os.pathsep + str(PROJECT)
sys.path[:0] = [str(PROJECT), str(UPSTREAM)]

from bagel_space.adapter import BagelAdapter
from bagel_space.capture import Capture

adapter = BagelAdapter()
print(torch.cuda.get_device_name(0), adapter.dtype)
print("Layers:", len(adapter.layers), "| Hidden size:", adapter.model.hidden_size)
from PIL import Image
from experiments.custom_utils import (
    prepare_pair, capture_pair, build_steering_vector, make_steering_vector,
    reduce_states, count_denoising_evaluations, steer_denoising,
)


## 3. Choose pairs and extraction settings
Choose any of the three counting folders, or replace `pairs` with your own positive/negative records.


In [ ]:
DATA = PROJECT / "inputs/counting/one_to_two_cross_branch/build_pairs.json"
pairs = json.loads(DATA.read_text(encoding="utf-8"))
OUTPUT = PROJECT / "outputs/custom-steering" / DATA.parent.name
OUTPUT.mkdir(parents=True, exist_ok=True)

alignment = "none"   # "none", "right_pad"
build_tokens = "last"      # "first", "last", "all", index, list, slice
layers = "all"
token_reduction = "mean"
SEED = 42
print("Pairs:", len(pairs))
print("Positive:", pairs[0]["positive"])
print("Negative:", pairs[0]["negative"])


## 4. Inspect tokenization
Extraction includes BOS and no EOS; right padding appends visible spaces.


In [ ]:
pair = prepare_pair(
    adapter, pairs[0]["positive"], pairs[0]["negative"], alignment=alignment,
)
for side, ids in pair["token_ids"].items():
    print(side, adapter.tokenizer.convert_ids_to_tokens(ids))
print("Padding:", pair["padding"])


## 5. Capture and select hidden states
Frozen understanding route; each selected layer stays separate.


In [ ]:
positive, negative = {}, {}
token_audit = []
for index, texts in enumerate(pairs):
    pair = prepare_pair(adapter, texts["positive"], texts["negative"], alignment=alignment)
    captured = capture_pair(adapter, pair, layers=layers)
    result = build_steering_vector(
        captured, tokens=build_tokens, token_reduction=token_reduction,
        layer_reduction="none", direction_method="difference",
    )
    for layer in result["vectors"]:
        positive.setdefault(layer, []).append(result["positive"][layer])
        negative.setdefault(layer, []).append(result["negative"][layer])
    token_audit.append(pair)
    print(index + 1, "/", len(pairs))

positive = {layer: torch.stack(states) for layer, states in positive.items()}
negative = {layer: torch.stack(states) for layer, states in negative.items()}
differences = {layer: positive[layer] - negative[layer] for layer in positive}
selected_layers = list(positive)
print("Layer:", selected_layers[0], "| Shape:", differences[selected_layers[0]].shape)


## 6. Construct vectors
Average pair differences independently at each layer.


In [ ]:
pair_reduction = "mean"
vectors = {layer: reduce_states(states, pair_reduction) for layer, states in differences.items()}
method = "mean_difference"
normalization = "none"
for layer, vector in vectors.items():
    print("Layer:", layer, "| Shape:", tuple(vector.shape), "| Norm:", vector.norm().item())


### RepE - later
Reserved for another estimator using step 5 states.


### ITI - later
Reserved for another estimator using step 5 states.


### Optional: unit normalization
Skip to keep the original mean-difference scale.


In [ ]:
vectors = {layer: vector / vector.norm(dim=-1, keepdim=True) for layer, vector in vectors.items()}
normalization = "unit"


## 7. Inspect the operation
First pair subtraction and the final vector, six coordinates.


In [ ]:
layer = selected_layers[0]
print("Positive:", positive[layer][0].reshape(-1)[:6])
print("Negative:", negative[layer][0].reshape(-1)[:6])
print("Difference:", differences[layer][0].reshape(-1)[:6])
print("Mean difference:", vectors[layer].reshape(-1)[:6])


## 8. Save vectors


In [ ]:
RUN_OUTPUT = OUTPUT / method
RUN_OUTPUT.mkdir(parents=True, exist_ok=True)
metadata = dict(
    **adapter.metadata(), dataset=str(DATA.relative_to(PROJECT)), n_pairs=len(pairs),
    method=method, normalization=normalization, alignment=alignment,
    tokens=repr(build_tokens), layers=selected_layers,
    token_reduction=getattr(token_reduction, "__name__", token_reduction),
    pair_reduction=getattr(pair_reduction, "__name__", pair_reduction),
    capture="text-only understanding block outputs; BOS, no EOS", seed=SEED,
)
torch.save(dict(vectors=vectors, metadata=metadata), RUN_OUTPUT / "steering_vectors.pt")
torch.save(
    dict(positive=positive, negative=negative, differences=differences),
    RUN_OUTPUT / "hidden_states.pt",
)
(RUN_OUTPUT / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
(RUN_OUTPUT / "pairs.json").write_text(json.dumps(pairs, indent=2), encoding="utf-8")
(RUN_OUTPUT / "token_audit.json").write_text(json.dumps(token_audit, indent=2), encoding="utf-8")
print("Saved:", RUN_OUTPUT)


## 9. Download vectors


In [ ]:
from google.colab import files

archive = shutil.make_archive("/content/bagel-steering-build", "zip", RUN_OUTPUT)
files.download(archive)


## 10. Generate baseline
Reuse the same prompt, seed, and sampling settings for steering.


In [ ]:
prompt = "A studio photograph of one yellow rubber duck on a plain blue background."
generation_kwargs = dict(steps=50, guidance=1.0, seed=SEED)
with count_denoising_evaluations(adapter) as baseline_trace:
    baseline = adapter.generate(prompt, **generation_kwargs)
total_evaluations = baseline_trace["evaluations"]
print("Conditional evaluations:", total_evaluations)
display(baseline)


## 11. Apply steering
Text edits affect prompt prefill once; `iterations` selects visual denoising evaluations. Special tokens stay untouched.


In [ ]:
application_layers = [6]
iterations = "all"
target = "text"       # "text", "visual", "text+visual"
tokens = "all"       # "first", "last", "all", index, list, slice
operation = "add"    # "add", "component", callable
alpha = 0.1

rule = dict(tokens=tokens, operation=operation, alpha=alpha)
rules = {
    "text": dict(text=rule, visual=None),
    "visual": dict(text=None, visual=rule),
    "text+visual": dict(text=rule, visual=rule),
}[target]
with steer_denoising(
    adapter, vectors=vectors, total_evaluations=total_evaluations,
    layers=application_layers, iterations=iterations, **rules,
) as steering_trace:
    steered = adapter.generate(prompt, **generation_kwargs)
print("Visual evaluations:", steering_trace["evaluations"])
print("Layer/modality injections:", len(steering_trace["injections"]))


### Optional: inspect the edit


In [ ]:
for name, values in steering_trace["preview"].items():
    print(name, "| Position:", values["position"])
    print("Before:", values["before"])
    print("After:", values["after"])


### Compare images
Left: baseline. Right: steering.


In [ ]:
width, height = baseline.size
comparison = Image.new("RGB", (2 * width, height))
comparison.paste(baseline, (0, 0))
comparison.paste(steered, (width, 0))
display(comparison)


## 12. Save and download images


In [ ]:
baseline.save(RUN_OUTPUT / "baseline.png")
steered.save(RUN_OUTPUT / "steered.png")
comparison.save(RUN_OUTPUT / "comparison.png")
torch.save(
    dict(prompt=prompt, generation=generation_kwargs, layers=application_layers,
         iterations=iterations, rules=rules, trace=steering_trace, vector_metadata=metadata),
    RUN_OUTPUT / "application.pt",
)
archive = shutil.make_archive("/content/bagel-steering-application", "zip", RUN_OUTPUT)
files.download(archive)


## 13. Optional: compare strengths
One row; zero uses the baseline. Text uses prefill, and visual uses the selected evaluations.


In [ ]:
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
from experiments.custom_utils import count_denoising_evaluations, steer_denoising

prompt = "A studio photograph of one yellow rubber duck on a plain blue background."
generation_kwargs = dict(steps=50, guidance=1.0, seed=SEED)

target = "text"       # "visual", "text", "text+visual"
operation = "add"       # "add", "component"
tokens = "all"       # "first", "last", "all", index, list, slice
application_layers = [6]
iterations = "all"
alphas = [-9, -5, -2, -1, 1, 2, 5, 9]

with count_denoising_evaluations(adapter) as trace:
    baseline = adapter.generate(prompt, **generation_kwargs)

results = {0: baseline}

for alpha in alphas:
    rule = dict(tokens=tokens, operation=operation, alpha=alpha)
    rules = {
        "visual": dict(text=None, visual=rule),
        "text": dict(text=rule, visual=None),
        "text+visual": dict(text=rule, visual=rule),
    }[target]

    with steer_denoising(
        adapter, vectors=vectors, total_evaluations=trace["evaluations"],
        layers=application_layers, iterations=iterations, **rules,
    ):
        results[alpha] = adapter.generate(prompt, **generation_kwargs)

    print(f"{target} | {operation} | alpha={alpha:+g} completed")

order = [-9, -5, -2, -1, 0, 1, 2, 5, 9]
size, header = 320, 90
row = Image.new("RGB", (size * len(order), size + header), "white")
draw = ImageDraw.Draw(row)
font = ImageFont.load_default(size=38)

for index, alpha in enumerate(order):
    x = index * size
    label = "Baseline" if alpha == 0 else f"alpha = {alpha:+g}"
    color = "#DCEBFA" if alpha == 0 else "#F3F4F6"
    draw.rectangle((x, 0, x + size - 1, header - 1), fill=color)
    draw.text(
        (x + size // 2, header // 2), label,
        fill="#172033", font=font, anchor="mm",
    )
    row.paste(
        results[alpha].resize((size, size), Image.Resampling.LANCZOS),
        (x, header),
    )

for index in range(1, len(order)):
    x = index * size
    draw.line((x, 0, x, row.height - 1), fill="#606875", width=3)

draw.line((0, header, row.width - 1, header), fill="#606875", width=3)

destination = OUTPUT / method / "rows"
destination.mkdir(parents=True, exist_ok=True)
filename = destination / f"{target}_{operation}.png"
row.save(filename)

print(method, "|", target, "|", operation)
print("Tokens:", tokens, "| Layers:", application_layers, "| Iterations:", iterations)
print("Prompt:", prompt, "| Generation:", generation_kwargs)
print("Saved:", filename)
display(row)
